# Pandas Aggregationen: Lagemaße, Gruppen und Top-N
### Unterrichtsmaterial | Julia Brutskaya

---
**Kontext:** Kurs-Umfrage der Datenwald-Akademie<br>
**Bibliotheken:** `pandas`

<img src="png%20Dateien/drilling_modus.png" alt="Modus" width="150"> <img src="png%20Dateien/drilling_median.png" alt="Median" width="150"> <img src="png%20Dateien/drilling_mittelwert.png" alt="Mittelwert" width="150">

Heute Vormittag hast du Modus, Median und Mittelwert mit Stift und Taschenrechner bestimmt. Jetzt berechnest du dieselben Werte mit Pandas und wertest danach ganze Gruppen auf einmal aus.

Die Igel-Drillinge sind wieder dabei:

| Drilling | Was er tut | In Pandas |
|---|---|---|
| **Modus** (gelber Schal) | macht, was die meisten machen | `.mode()` |
| **Median** (grüner Schal) | stellt sich in die Mitte der Reihe | `.median()` |
| **Mittelwert** (roter Schal) | rechnet jeden mit ein | `.mean()` |

Herr Bauchgefühl stellt wieder Behauptungen auf, und der Ausreißer hat heute seinen großen Auftritt.

| Block | Inhalt |
|---|---|
| A | Der Modus |
| B | Median und Mittelwert |
| C | Der Ausreißer und der Überblick |
| D | Gruppen auswerten mit `groupby` |
| E | Sortieren und Top-N |
| F | Weitere Übungen |

---
# Block A – Der Modus

## 1 – Die Kurs-Umfrage laden

Du arbeitest mit derselben Umfrage wie gestern. Die T-Shirt-Größen werden gleich als geordnete Kategorie angelegt, damit Pandas die Reihenfolge S < M < L < XL kennt.

In [1]:
import pandas as pd

# Kurs-Umfrage der Datenwald-Akademie: eine Zeile pro befragter Person
zeilen = [
    (101, "Vormittag", "Kaffee", "M", 0, 25), (102, "Abend", "Tee", "S", 1, 35),
    (103, "Vormittag", "Mate", "L", 0, 18), (104, "Vormittag", "Kaffee", "L", 2, 42),
    (105, "Abend", "Wasser", "M", 0, 30), (106, "Vormittag", "Tee", "M", 1, 28),
    (107, "Abend", "Kaffee", "XL", 0, 55), (108, "Vormittag", "Kaffee", "S", 3, 12),
    (109, "Vormittag", "Mate", "M", 1, 38), (110, "Abend", "Tee", "M", 0, 115),
    (111, "Vormittag", "Kaffee", "L", 0, 32), (112, "Abend", "Mate", "L", 2, 45),
    (113, "Vormittag", "Wasser", "S", 1, 22), (114, "Vormittag", "Kaffee", "M", 0, 35),
    (115, "Abend", "Tee", "L", 1, 48), (116, "Vormittag", "Tee", "M", 0, 30),
    (117, "Abend", "Kaffee", "M", 3, 65), (118, "Vormittag", "Mate", "XL", 1, 40),
    (119, "Abend", "Wasser", "S", 0, 52), (120, "Vormittag", "Kaffee", "L", 2, 25),
]
spalten = ["teilnehmer_id", "kurszeit", "getraenk", "tshirt", "haustiere", "anfahrt_min"]
umfrage = pd.DataFrame(zeilen, columns=spalten)

# T-Shirt-Größen als geordnete Kategorie (wie in Notebook 1)
groessen = ["S", "M", "L", "XL"]
umfrage["tshirt"] = pd.Categorical(umfrage["tshirt"], categories=groessen, ordered=True)

umfrage.head()

,teilnehmer_id,kurszeit,getraenk,tshirt,haustiere,anfahrt_min
0,101,Vormittag,Kaffee,M,0,25
1,102,Abend,Tee,S,1,35
2,103,Vormittag,Mate,L,0,18
3,104,Vormittag,Kaffee,L,2,42
4,105,Abend,Wasser,M,0,30


## 2 – Der Modus mit `mode()`

<img src="png%20Dateien/drilling_modus.png" alt="Modus" width="150">

> **Herr Bauchgefühl:** „Das Lieblingsgetränk unserer Teilnehmenden? Wasser natürlich. Wasser trinkt jeder."

Der Modus ist die Ausprägung, die am häufigsten vorkommt. In Pandas liefert ihn `.mode()`.

In [2]:
print(umfrage["getraenk"].mode())

0    Kaffee
Name: getraenk, dtype: str


Das Ergebnis ist keine einzelne Ausprägung, sondern eine **Series**. Der Grund: Es kann mehrere Modi geben. Den ersten holst du mit `[0]` heraus.

In [3]:
print("Modus Getränk:", umfrage["getraenk"].mode()[0])
print("Modus T-Shirt:", umfrage["tshirt"].mode()[0])

Modus Getränk: Kaffee
Modus T-Shirt: M


Die Gegenprobe kennst du von gestern: In der Häufigkeitstabelle steht der Modus in der Zeile mit der größten Häufigkeit.

In [4]:
print(umfrage["getraenk"].value_counts())

getraenk
Kaffee    8
Tee       5
Mate      4
Wasser    3
Name: count, dtype: int64


Kaffee gewinnt mit 8 Stimmen, Wasser steht auf dem letzten Platz.

<img src="png%20Dateien/herr_bauchgefuehl_verdutzt.png" alt="Herr Bauchgefühl stutzt" width="130">

Der Modus heißt „Kaffee", nicht 8. Die 8 ist nur die Häufigkeit, mit der Kaffee gewinnt.

## 3 – Mehrere Modi und die Modalklasse

Bei Messwerten hilft der Modus oft nicht weiter. Bei der Anfahrt gibt es gleich drei davon.

In [5]:
print(umfrage["anfahrt_min"].mode().tolist())

[25, 30, 35]


25, 30 und 35 Minuten kommen je zweimal vor, alle anderen Werte einmal. Deshalb bildest du bei stetigen Merkmalen Klassen mit `pd.cut`. Die Klasse mit den meisten Werten ist die **Modalklasse**.

In [6]:
klassen = pd.cut(umfrage["anfahrt_min"], bins=[0, 20, 40, 60, 120])

print(klassen.value_counts().sort_index())
print()
print("Modalklasse:", klassen.mode()[0])

anfahrt_min
(0, 20]       2
(20, 40]     11
(40, 60]      5
(60, 120]     2
Name: count, dtype: int64

Modalklasse: (20, 40]


Die Modalklasse ist „über 20 bis 40 Minuten" mit 11 von 20 Personen.

### Übung 1

- **a)** Bestimme den Modus von `kurszeit`.
- **b)** Bestimme den Modus von `haustiere`. Prüfe das Ergebnis mit `value_counts()`.
- **c)** Teile `anfahrt_min` in die Klassen (0, 30], (30, 60] und (60, 120] ein. Welche ist die Modalklasse?

In [91]:
# a)
print("Modus:", umfrage["kurszeit"].mode()[0])

Modus: Vormittag


In [92]:
# b)
print(umfrage["haustiere"].mode())
print()
print(umfrage["haustiere"].value_counts())


0    0
Name: haustiere, dtype: int64

haustiere
0    9
1    6
2    3
3    2
Name: count, dtype: int64


In [94]:
# c)
klassen_neu = pd.cut(umfrage["anfahrt_min"], bins=[0, 30, 60, 120])
print()
print(klassen_neu.mode())
print(klassen_neu.value_counts().sort_index())


0    (30, 60]
Name: anfahrt_min, dtype: category
Categories (3, interval[int64, right]): [(0, 30] < (30, 60] < (60, 120]]
anfahrt_min
(0, 30]       8
(30, 60]     10
(60, 120]     2
Name: count, dtype: int64


---
# Block B – Median und Mittelwert

## 4 – Der Median mit `median()`

<img src="png%20Dateien/drilling_median.png" alt="Median" width="150">

Der Median ist der Wert in der Mitte der sortierten Reihe. Vormittags hast du sortiert und abgezählt, Pandas erledigt beides in einem Schritt.

In [10]:
print("Median Anfahrt:  ", umfrage["anfahrt_min"].median())
print("Median Haustiere:", umfrage["haustiere"].median())

Median Anfahrt:   35.0
Median Haustiere: 1.0


Zum Nachvollziehen die sortierte Reihe wie am Vormittag: Bei 20 Werten liegt die Mitte zwischen dem 10. und dem 11. Wert.

In [11]:
sortiert = umfrage["anfahrt_min"].sort_values().tolist()

print(sortiert)
print("10. und 11. Wert:", sortiert[9], "und", sortiert[10])

[12, 18, 22, 25, 25, 28, 30, 30, 32, 35, 35, 38, 40, 42, 45, 48, 52, 55, 65, 115]
10. und 11. Wert: 35 und 35


Python zählt ab 0. Der 10. Wert steht deshalb an Position 9, der 11. an Position 10. Beide sind 35, der Median ist also 35 Minuten.

## 5 – Der Median eines ordinalen Merkmals

> **Herr Bauchgefühl:** „Ich habe die T-Shirt-Größen durchnummeriert und gemittelt. Die durchschnittliche Größe ist 2,3. Bestellt also bitte Größe M Komma drei."

So hat er gerechnet: S = 1, M = 2, L = 3, XL = 4. Pandas speichert zu jeder Kategorie eine solche Nummer (ab 0), abrufbar über `.cat.codes`.

In [12]:
nummern = umfrage["tshirt"].cat.codes + 1
print("Mittelwert der Nummern:", nummern.mean())

Mittelwert der Nummern: 2.3


Die 2,3 lässt sich ausrechnen, aber nicht bestellen. Bei ordinalen Merkmalen sind die Abstände zwischen den Stufen nicht gleich, deshalb gibt es keinen Mittelwert.

Der Median braucht nur die Reihenfolge. Du sortierst die Spalte und siehst dir die beiden mittleren Plätze an.

In [ ]:
tshirt_sortiert = umfrage["tshirt"].sort_values().reset_index(drop=True)

print(tshirt_sortiert.tolist())
print("Die beiden mittleren Plätze:", tshirt_sortiert.iloc[[9, 10]].tolist())
print(umfrage["tshirt"].count())
print(umfrage.shape)
# print(tshirt_sortiert.median())
# print(umfrage["tshirt"].median())

['S', 'S', 'S', 'S', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'M', 'L', 'L', 'L', 'L', 'L', 'L', 'XL', 'XL']
Die beiden mittleren Plätze: ['M', 'M']
20
(20, 6)
<bound method Series.median of 0      M
1      S
2      L
3      L
4      M
5      M
6     XL
7      S
8      M
9      M
10     L
11     L
12     S
13     M
14     L
15     M
16     M
17    XL
18     S
19     L
Name: tshirt, dtype: category
Categories (4, str): ['S' < 'M' < 'L' < 'XL']>


Auf beiden mittleren Plätzen steht M, der Median ist also M. Das funktioniert nur, weil `tshirt` eine geordnete Kategorie ist. Als normaler Text würde Pandas alphabetisch sortieren (L, M, S, XL).

`reset_index(drop=True)` nummeriert die sortierte Reihe neu durch, damit die Positionen 9 und 10 wirklich die Mitte sind.

## 6 – Der Mittelwert mit `mean()`

<img src="png%20Dateien/drilling_mittelwert.png" alt="Mittelwert" width="150">

> **Herr Bauchgefühl:** „Im Schnitt 0,9 Haustiere pro Person? Das kann nicht stimmen. Niemand hat 0,9 Haustiere."

Der Mittelwert ist die Summe aller Werte geteilt durch ihre Anzahl.

In [14]:
summe = umfrage["anfahrt_min"].sum()
anzahl = umfrage["anfahrt_min"].count()

print("Summe:", summe, "| Anzahl:", anzahl)
print("Von Hand:", summe / anzahl)
print("Mit mean():", umfrage["anfahrt_min"].mean())

Summe: 792 | Anzahl: 20
Von Hand: 39.6
Mit mean(): 39.6


In [15]:
print("Mittelwert Haustiere:", umfrage["haustiere"].mean())

Mittelwert Haustiere: 0.9


Die Rechnung stimmt. Der Mittelwert muss kein Wert sein, der in den Daten vorkommt. Bei den Haustieren liefern die drei Lagemaße drei verschiedene Antworten:

In [16]:
print("Modus:     ", umfrage["haustiere"].mode()[0])
print("Median:    ", umfrage["haustiere"].median())
print("Mittelwert:", umfrage["haustiere"].mean())

Modus:      0
Median:     1.0
Mittelwert: 0.9


## 7 – Das gewichtete Mittel aus der Häufigkeitstabelle

Vormittags hast du den Mittelwert auch aus der Häufigkeitstabelle berechnet: jede Ausprägung mal ihre Häufigkeit, dann die Summe durch die Anzahl der Personen.

`reset_index()` macht aus dem Ergebnis von `value_counts()` einen DataFrame mit zwei Spalten: `haustiere` und `count`.

In [17]:
tabelle = umfrage["haustiere"].value_counts().sort_index().reset_index()
tabelle["produkt"] = tabelle["haustiere"] * tabelle["count"]   # Haustiere mal Personen
tabelle

,haustiere,count,produkt
0,0,9,0
1,1,6,6
2,2,3,6
3,3,2,6


In [18]:
print("Haustiere gesamt:", tabelle["produkt"].sum())
print("Personen gesamt: ", tabelle["count"].sum())
print("Gewichtetes Mittel:", tabelle["produkt"].sum() / tabelle["count"].sum())

Haustiere gesamt: 18
Personen gesamt:  20
Gewichtetes Mittel: 0.9


18 Haustiere verteilt auf 20 Personen ergeben 0,9. Das ist dasselbe Ergebnis wie mit `mean()`.

### Übung 2

- **a)** Berechne Median und Mittelwert von `haustiere`.
- **b)** Berechne Median und Mittelwert der Anfahrt nur für den Vormittagskurs.
- **c)** Sortiere die Spalte `haustiere` und gib die beiden mittleren Werte aus. Passt das zum Median aus a)?

In [103]:
# a)
print(umfrage["haustiere"].median())
print(umfrage["haustiere"].mean())


1.0
0.9


In [104]:
# b)
vormittag = umfrage[umfrage["kurszeit"] == "Vormittag"]
print(vormittag["anfahrt_min"].median())
print(round(vormittag["anfahrt_min"].mean(), 1))

29.0
28.9


In [ ]:
# c)
tiere_sort = umfrage["haustiere"].sort_values().reset_index(drop=True)
print(tiere_sort.tolist())
print(tiere_sort.iloc[[9, 10]].tolist()) # Mittelwert

print(tiere_sort.mean()) # Durchschnitt


[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3]
[1, 1]
0.9


---
# Block C – Der Ausreißer und der Überblick

## 8 – Der Ausreißer zieht am Mittelwert

<img src="png%20Dateien/der_ausreisser.png" alt="Der Ausreißer, der Kater" width="170">

> **Herr Bauchgefühl:** „Der Mittelwert ist 39,6 Minuten. Also braucht die eine Hälfte von euch länger als 39,6 Minuten und die andere kürzer."

Du zählst nach. Eine Bedingung ergibt `True` oder `False`, und `.sum()` zählt die `True`-Werte.

In [22]:
mittel = umfrage["anfahrt_min"].mean()

print("Länger als der Mittelwert:", (umfrage["anfahrt_min"] > mittel).sum(), "Personen")
print("Kürzer als der Mittelwert:", (umfrage["anfahrt_min"] < mittel).sum(), "Personen")

Länger als der Mittelwert: 8 Personen
Kürzer als der Mittelwert: 12 Personen


Nur 8 von 20 brauchen länger als der Mittelwert. In zwei Hälften teilt der Median, und der liegt bei 35 Minuten.

Schuld ist der Ausreißer mit seinen 115 Minuten. Was passiert, wenn du ihn herausnimmst?

In [23]:
ohne = umfrage[umfrage["anfahrt_min"] < 100]

vergleich = pd.DataFrame({
    "mit Ausreißer":  [umfrage["anfahrt_min"].mean(), umfrage["anfahrt_min"].median()],
    "ohne Ausreißer": [ohne["anfahrt_min"].mean(), ohne["anfahrt_min"].median()],
}, index=["Mittelwert", "Median"])

vergleich.round(1)

,mit Ausreißer,ohne Ausreißer
Mittelwert,39.6,35.6
Median,35.0,35.0


Ein einziger von 20 Werten verschiebt den Mittelwert um 4 Minuten. Der Median rührt sich nicht.

- Der Median ist **robust** gegen Ausreißer.
- Der Mittelwert ist **ausreißerempfindlich**.

**Wichtig:** Hier wurde der Ausreißer nur zum Vergleich herausgefiltert. `umfrage` selbst enthält weiter alle 20 Personen. Einen echten Wert löscht man nicht einfach aus den Daten.

## 9 – Mittelwert und Median vergleichen

Liegen Mittelwert und Median auseinander, ist die Verteilung schief.

| Vergleich | Form |
|---|---|
| Mittelwert größer als Median | rechtsschief |
| Mittelwert kleiner als Median | linksschief |
| beide ungefähr gleich | symmetrisch |

In [24]:
mittel = umfrage["anfahrt_min"].mean()
median = umfrage["anfahrt_min"].median()

print("Mittelwert:", mittel)
print("Median:    ", median)
print("Abstand:   ", round(mittel - median, 1), "Minuten")

Mittelwert: 39.6
Median:     35.0
Abstand:    4.6 Minuten


Der Mittelwert liegt 4,6 Minuten über dem Median. Die Anfahrt ist **rechtsschief**: Wenige sehr lange Wege ziehen den Mittelwert nach oben.

## 10 – Der Überblick mit `describe()`

`describe()` fasst alle Zahlenspalten auf einmal zusammen.

In [25]:
umfrage[["haustiere", "anfahrt_min"]].describe()

,haustiere,anfahrt_min
count,20.000000,20.000000
mean,0.900000,39.600000
std,1.020836,22.020087
min,0.000000,12.000000
25%,0.000000,27.250000
50%,1.000000,35.000000
75%,1.250000,45.750000
max,3.000000,115.000000


| Zeile | Bedeutung |
|---|---|
| `count` | Anzahl der Werte |
| `mean` | Mittelwert |
| `min`, `max` | kleinster und größter Wert |
| `50%` | der Median |
| `std`, `25%`, `75%` | Streuung und Quartile, Thema von Notebook 3 |

### Übung 3

- **a)** Wie viele Personen im Abendkurs brauchen länger als der Mittelwert des Abendkurses?
- **b)** Berechne für den Vormittagskurs Mittelwert und Median der Anfahrt. Welche Form hat die Verteilung?
- **c)** Lass dir mit `describe()` die Anfahrt des Vormittagskurses zusammenfassen. Lies die kürzeste und die längste Anfahrt ab.

In [ ]:
# a)
abend = umfrage[umfrage["kurszeit"] == "Abend"]
mittel_abend = abend["anfahrt_min"].mean()

print(mittel_abend)
print((abend["anfahrt_min"] > mittel_abend).sum(), "von", len(abend)) # 2: 65, 115


55.625
2 von 8


In [ ]:
# b)
vormittag = umfrage[umfrage["kurszeit"] == "Vormittag"]

print(round(vormittag["anfahrt_min"].mean(), 1))
print(vormittag["anfahrt_min"].median())

# symmetrisch, keine Ausreißer im Vormittagskurs

28.9
29.0


In [116]:
# c)
print(vormittag["anfahrt_min"].describe())

count    12.000000
mean     28.916667
std       9.090038
min      12.000000
25%      24.250000
50%      29.000000
75%      35.750000
max      42.000000
Name: anfahrt_min, dtype: float64


---
# Block D – Gruppen auswerten mit `groupby`

## 11 – Gruppen bilden mit `groupby()`

Bisher hast du für jede Gruppe einzeln gefiltert: erst den Vormittagskurs, dann den Abendkurs. `groupby()` erledigt das für alle Gruppen auf einmal, in drei Schritten:

1. **Aufteilen:** Die Zeilen werden nach einer Spalte in Gruppen zerlegt.
2. **Anwenden:** Auf jede Gruppe wird dieselbe Berechnung angewendet.
3. **Zusammenführen:** Die Ergebnisse landen in einer neuen Tabelle.

In [29]:
# Mittelwert der Anfahrt pro Kurszeit
print(umfrage.groupby("kurszeit")["anfahrt_min"].mean())

kurszeit
Abend        55.625000
Vormittag    28.916667
Name: anfahrt_min, dtype: float64


So liest du die Zeile: `groupby("kurszeit")` bildet die Gruppen, `["anfahrt_min"]` wählt die Spalte, `.mean()` ist die Berechnung.

Jede Methode, die du von einer einzelnen Spalte kennst, funktioniert auch nach `groupby`.

In [30]:
# Median der Anfahrt pro Kurszeit
print(umfrage.groupby("kurszeit")["anfahrt_min"].median())

kurszeit
Abend        50.0
Vormittag    29.0
Name: anfahrt_min, dtype: float64


In [31]:
# Anzahl der Personen pro Getränk
print(umfrage.groupby("getraenk")["teilnehmer_id"].count())

getraenk
Kaffee    8
Mate      4
Tee       5
Wasser    3
Name: teilnehmer_id, dtype: int64


## 12 – Mehrere Kennzahlen mit `agg()`

Mit `.agg()` berechnest du mehrere Kennzahlen in einem Schritt. Du übergibst eine Liste mit den Namen der Berechnungen.

In [32]:
gruppen = umfrage.groupby("kurszeit")["anfahrt_min"].agg(["count", "sum", "mean", "median"])
gruppen

,count,sum,mean,median
kurszeit,,,,
Abend,8,445,55.625000,50.0
Vormittag,12,347,28.916667,29.0


Mit `.round(1)` wird die Tabelle lesbarer.

In [33]:
gruppen.round(1)

,count,sum,mean,median
kurszeit,,,,
Abend,8,445,55.6,50.0
Vormittag,12,347,28.9,29.0


## 13 – Die Falle: Mittelwerte mitteln

<img src="png%20Dateien/herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="140">

> **Herr Bauchgefühl:** „Vormittags 28,9 Minuten, abends 55,6. Dann liegt die ganze Akademie bei (28,9 + 55,6) / 2, also bei 42,3 Minuten."

In [ ]:
print("Mittel der beiden Kursmittelwerte:", round(gruppen["mean"].mean(), 1)) # falsch
print("Mittelwert aller 20 Personen:     ", umfrage["anfahrt_min"].mean())

Mittel der beiden Kursmittelwerte: 42.3
Mittelwert aller 20 Personen:      39.6


42,3 ist falsch, denn die Kurse sind verschieden groß (12 und 8 Personen). Jeder Kursmittelwert muss mit seiner Personenzahl gewichtet werden.

In [118]:
print(gruppen.round(1))
gewichtet = (gruppen["count"] * gruppen["mean"]).sum() / gruppen["count"].sum()
print("Gewichtetes Mittel:", round(gewichtet, 1))

           count  sum  mean  median
kurszeit                           
Abend          8  445  55.6    50.0
Vormittag     12  347  28.9    29.0
Gewichtetes Mittel: 39.6


Das ist wieder der Mittelwert aller 20 Personen. Einfach mitteln darfst du Mittelwerte nur, wenn die Gruppen gleich groß sind.

## 14 – Benannte Aggregationen

Für einen Bericht brauchst du oft Kennzahlen aus verschiedenen Spalten und mit sprechenden Namen. Dafür gibt es die benannte Schreibweise:

`neuer_name = ("spalte", "berechnung")`

In [36]:
bericht = umfrage.groupby("kurszeit").agg(
    personen=("teilnehmer_id", "count"),
    anfahrt_mittel=("anfahrt_min", "mean"),
    anfahrt_median=("anfahrt_min", "median"),
    haustiere_gesamt=("haustiere", "sum"),
)
bericht.round(1)

,personen,anfahrt_mittel,anfahrt_median,haustiere_gesamt
kurszeit,,,,
Abend,8,55.6,50.0,7
Vormittag,12,28.9,29.0,11


### Übung 4

- **a)** Berechne den Median der Anfahrt für jedes Getränk.
- **b)** Berechne für jede Kurszeit die Anzahl, den Mittelwert und den Median der Spalte `haustiere`. Nutze `agg()` mit einer Liste.
- **c)** Erstelle mit benannten Aggregationen einen Bericht pro Getränk: `personen` (Anzahl) und `anfahrt_mittel` (Mittelwert der Anfahrt). Runde auf eine Nachkommastelle.

In [119]:
# a)
print(umfrage.groupby("getraenk")["anfahrt_min"].median())

getraenk
Kaffee    33.5
Mate      39.0
Tee       35.0
Wasser    30.0
Name: anfahrt_min, dtype: float64


In [120]:
# b)
umfrage.groupby("kurszeit")["haustiere"].agg(["count", "mean", "median"]).round(2)

,count,mean,median
kurszeit,,,
Abend,8,0.88,0.5
Vormittag,12,0.92,1.0


In [121]:
# c)
getr_ber = umfrage.groupby("getraenk").agg(
    personen=("teilnehmer_id", "count"),
    anfahrt_mittel=("anfahrt_min", "mean"),
)
getr_ber.round(1)

,personen,anfahrt_mittel
getraenk,,
Kaffee,8,36.4
Mate,4,35.2
Tee,5,51.2
Wasser,3,34.7


---
# Block E – Sortieren und Top-N

## 15 – Sortieren und die ersten N Zeilen

> **Herr Bauchgefühl:** „Die längsten Anfahrten? Das sind bestimmt die Frühaufsteher aus dem Vormittagskurs."

Eine **Top-N-Analyse** zeigt die N größten oder kleinsten Einträge. Der direkte Weg: absteigend sortieren und mit `head(N)` die ersten N Zeilen nehmen.

In [40]:
# Die 5 längsten Anfahrten
umfrage.sort_values("anfahrt_min", ascending=False).head(5)

,teilnehmer_id,kurszeit,getraenk,tshirt,haustiere,anfahrt_min
9,110,Abend,Tee,M,0,115
16,117,Abend,Kaffee,M,3,65
6,107,Abend,Kaffee,XL,0,55
18,119,Abend,Wasser,S,0,52
14,115,Abend,Tee,L,1,48


Alle fünf kommen aus dem Abendkurs. Herr Bauchgefühl stutzt.

## 16 – `nlargest()` und `nsmallest()`

Für Top-N gibt es eine Abkürzung, die Sortieren und Abschneiden verbindet.

| Befehl | Ergebnis |
|---|---|
| `df.nlargest(N, "spalte")` | die N Zeilen mit den größten Werten |
| `df.nsmallest(N, "spalte")` | die N Zeilen mit den kleinsten Werten |

In [41]:
# Die 3 längsten Anfahrten, nur ausgewählte Spalten
umfrage.nlargest(3, "anfahrt_min")[["teilnehmer_id", "kurszeit", "anfahrt_min"]]

,teilnehmer_id,kurszeit,anfahrt_min
9,110,Abend,115
16,117,Abend,65
6,107,Abend,55


In [42]:
# Die 3 kürzesten Anfahrten
umfrage.nsmallest(3, "anfahrt_min")[["teilnehmer_id", "kurszeit", "anfahrt_min"]]

,teilnehmer_id,kurszeit,anfahrt_min
7,108,Vormittag,12
2,103,Vormittag,18
12,113,Vormittag,22


**Achtung bei Gleichstand:** Zwei Personen haben 3 Haustiere, drei Personen haben 2. Wer gehört zu den „Top 3"? `nlargest` nimmt die Zeilen, die zuerst in der Tabelle stehen. Mit `keep="all"` bleiben alle erhalten, die gleichauf liegen.

In [123]:
umfrage.nlargest(3, "haustiere", keep="all")[["teilnehmer_id", "haustiere"]]

,teilnehmer_id,haustiere
7,108,3
16,117,3
3,104,2
11,112,2
19,120,2


## 17 – Top-N nach einer Gruppenauswertung

Oft willst du nicht einzelne Zeilen vergleichen, sondern Gruppen. Dann kombinierst du `groupby` mit dem Sortieren.

In [44]:
# Welche Getränkegruppe hat die längste mittlere Anfahrt?
anfahrt_pro_getraenk = umfrage.groupby("getraenk")["anfahrt_min"].mean()

print(anfahrt_pro_getraenk.sort_values(ascending=False).round(1))

getraenk
Tee       51.2
Kaffee    36.4
Mate      35.2
Wasser    34.7
Name: anfahrt_min, dtype: float64


Die Tee-Gruppe führt mit Abstand. Der Grund ist der Ausreißer: Die Person mit 115 Minuten trinkt Tee, und die Gruppe hat nur fünf Mitglieder. Bei kleinen Gruppen lohnt sich der Blick auf den Median besonders.

### Übung 5

- **a)** Zeige die 3 Personen mit den wenigsten Haustieren im Abendkurs. Nutze `keep="all"`.
- **b)** Zeige die 4 längsten Anfahrten des Vormittagskurses mit den Spalten `teilnehmer_id`, `getraenk` und `anfahrt_min`.
- **c)** Welche Getränkegruppe hat den höchsten **Median** der Anfahrt? Vergleiche mit dem Ergebnis aus Abschnitt 17.

In [124]:
# a)
abend = umfrage[umfrage["kurszeit"] == "Abend"]
abend.nsmallest(3, "haustiere", keep="all")[["teilnehmer_id", "haustiere"]]

,teilnehmer_id,haustiere
4,105,0
6,107,0
9,110,0
18,119,0


In [125]:
# b)
vormittag = umfrage[umfrage["kurszeit"] == "Vormittag"]
vormittag.nlargest(4, "anfahrt_min")[["teilnehmer_id", "getraenk", "anfahrt_min"]]

,teilnehmer_id,getraenk,anfahrt_min
3,104,Kaffee,42
17,118,Mate,40
8,109,Mate,38
13,114,Kaffee,35


In [127]:
# c)
median_pro_getr = umfrage.groupby("getraenk")["anfahrt_min"].median()
print(median_pro_getr.sort_values(ascending=False))

getraenk
Mate      39.0
Tee       35.0
Kaffee    33.5
Wasser    30.0
Name: anfahrt_min, dtype: float64


---
# Block F – Weitere Übungen

Jede Übung nennt den Block, zu dem sie gehört.

- **Übung 6 bis 9:** Gegenprobe zu deinen Handrechnungen vom Vormittag.
- **Übung 10:** Die Charts.
- **Übung 11 bis 14:** Die TechAcademy GmbH.
- **Übung 15 und 16:** Zusatz zu `groupby()` und Top-N (TechAcademy).

## 18 – Gegenprobe zum Vormittag

Lege deine Lösungen vom Vormittag daneben und vergleiche. Pandas schreibt Dezimalzahlen mit Punkt, im Skript stehen sie mit Komma.

### Übung 6 – Fehltage (zu Block A und B)

Vormittags war das Übung 4. Die Häufigkeitstabelle der Fehltage ist hier schon in eine Datenreihe umgewandelt: sechsmal 0, fünfmal 1 und so weiter. Führe die Zelle aus und berechne Mittelwert, Median und Modus.

In [48]:
# 6 Personen mit 0 Fehltagen, 5 mit 1, 4 mit 2, 3 mit 3 und 2 mit 5
fehltage = pd.Series([0] * 6 + [1] * 5 + [2] * 4 + [3] * 3 + [5] * 2)
print(fehltage.tolist())

[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 5, 5]


In [49]:
# Übung 6


### Übung 7 – Was verdient man in der Akademie? (zu Block C)

Vormittags war das Übung 6. Fünf Mitarbeitende verdienen 2.400 €, 2.600 €, 2.800 €, 3.000 € und 3.200 €. Herr Bauchgefühl verdient 14.000 €.

- **a)** Berechne Mittelwert und Median aller sechs Gehälter.
- **b)** Berechne beide Werte ohne Herrn Bauchgefühl.
- **c)** In der Stellenanzeige steht „Durchschnittsgehalt 4.667 €". Ist das ehrlich? Antworte als Kommentar.

In [50]:
gehalt = pd.Series([2400, 2600, 2800, 3000, 3200, 14000])

In [51]:
# a)


In [52]:
# b)


In [53]:
# c)


### Übung 8 – Der Abendkurs und der Ausreißer (zu Block C)

Vormittags war das Übung 9.

- **a)** Hole die Anfahrtszeiten des Abendkurses, gib sie sortiert aus und berechne Mittelwert und Median.
- **b)** Berechne beide Werte noch einmal ohne die 115 Minuten.
- **c)** Welches Lagemaß hat sich stärker verändert, und warum? Antworte als Kommentar.

In [54]:
# a)


In [55]:
# b)


In [56]:
# c)


### Übung 9 – Bonus: Wachstum im Schnitt (zu Block B)

Vormittags war das Übung 10. Für Wachstumsfaktoren nimmt man das **geometrische Mittel**: alle Faktoren multiplizieren und die n-te Wurzel ziehen. In Python schreibst du die n-te Wurzel als `** (1 / n)`.

Das Beispiel aus dem Skript: erst plus 50 %, dann minus 50 %.

In [57]:
faktoren = [1.5, 0.5]

produkt = faktoren[0] * faktoren[1]
geometrisches_mittel = produkt ** (1 / len(faktoren))

print("Durchschnittlicher Faktor:", round(geometrisches_mittel, 3))

Durchschnittlicher Faktor: 0.866


Der Faktor 0,866 bedeutet: im Schnitt minus 13,4 % pro Jahr.

Jetzt du: Die Anmeldungen sind erst um 20 % und im Jahr darauf um 80 % gestiegen. Herr Bauchgefühl sagt: „Im Schnitt 50 % pro Jahr." Rechne nach.

In [58]:
# Übung 9


## 19 – Die Charts

Die Datei `UK-top40-1964-1-2.tsv` kennst du aus Notebook 1. Sie muss im selben Ordner liegen wie dieses Notebook. Die Spalte `WoC` gibt an, wie viele Wochen ein Song schon in den Charts steht.

In [59]:
# sep="\t": Die Spalten sind in dieser Datei durch Tabulatoren getrennt
charts = pd.read_csv("csv & excel Dateien/UK-top40-1964-1-2.tsv", sep="\t")
charts.head()

,Pos,LW,Title,Artist,Publisher,Peak Pos,WoC
0,1,1,I WANT TO HOLD YOUR HAND,THE BEATLES,PARLOPHONE,1,5
1,2,6,GLAD ALL OVER,THE DAVE CLARK FIVE,COLUMBIA,2,7
2,3,2,SHE LOVES YOU,THE BEATLES,PARLOPHONE,1,19
3,4,3,YOU WERE MADE FOR ME,FREDDIE AND THE DREAMERS,COLUMBIA,3,9
4,5,9,TWENTY FOUR HOURS FROM TULSA,GENE PITNEY,UNITED ARTISTS,5,5


### Übung 10 – Wie lange bleibt ein Song in den Charts? (zu Block A bis C und E)

- **a)** Berechne für `WoC` Mittelwert, Median und Modus.
- **b)** Ist die Verteilung rechts- oder linksschief? Welches Lagemaß beschreibt einen typischen Song am besten?
- **c)** Zeige die fünf Songs mit den meisten Wochen. Nutze `keep="all"`.
- **d)** Sind das Ausreißer? Antworte als Kommentar.

In [60]:
# a)


In [61]:
# b)


In [62]:
# c)


In [63]:
# d)


## 20 – Zweiter Schauplatz: die TechAcademy GmbH

Führe die folgende Zelle aus. Sie erstellt zwei DataFrames: `pruefungen` (eine Zeile pro Prüfung) und `kursstatistik` (eine Zeile pro Kurs).

In [64]:
# Prüfungsergebnisse der TechAcademy GmbH
pruefungen = pd.DataFrame({
    "Teilnehmer": ["Mia Schulz", "Jonas Weber", "Lena Koch", "Tim Bauer", "Sara Yilmaz",
                   "Paul Richter", "Anna Stern", "Ben Klose", "Clara Vogt", "Max Huber",
                   "Mia Schulz", "Jonas Weber", "Clara Vogt", "Ben Klose", "Anna Stern"],
    "Kurs":       ["Python Grundlagen", "SQL für Einsteiger", "Python Grundlagen", "Data Analysis", "Linux Basics",
                   "SQL für Einsteiger", "Data Analysis", "Python Grundlagen", "Linux Basics", "Data Analysis",
                   "Data Analysis", "Python Grundlagen", "SQL für Einsteiger", "Linux Basics", "Linux Basics"],
    "Kursformat": ["Präsenz", "Online", "Präsenz", "Online", "Präsenz",
                   "Online", "Präsenz", "Online", "Präsenz", "Online",
                   "Online", "Präsenz", "Präsenz", "Online", "Online"],
    "Trainer":    ["Sandra Meier", "Tobias Lange", "Sandra Meier", "Michael Brandt", "Yara Özcan",
                   "Tobias Lange", "Michael Brandt", "Sandra Meier", "Yara Özcan", "Michael Brandt",
                   "Michael Brandt", "Sandra Meier", "Tobias Lange", "Yara Özcan", "Yara Özcan"],
    "Monat":      ["Januar", "Januar", "Januar", "Februar", "Februar",
                   "Februar", "März", "März", "März", "März",
                   "April", "April", "April", "April", "April"],
    "Note":       [2.0, 1.7, 3.3, 1.0, 2.7,
                   2.3, 1.3, 4.0, 2.0, 1.7,
                   1.3, 2.7, 2.0, 3.7, 4.3],
    "Bestanden":  [True, True, True, True, True,
                   True, True, False, True, True,
                   True, True, True, False, False],
})

# Kursstatistik der TechAcademy GmbH
kursstatistik = pd.DataFrame({
    "Kursname":    ["Python Grundlagen", "SQL für Einsteiger", "Data Analysis",
                    "Linux Basics", "Power BI", "Algorithmen",
                    "Python Fortgeschrittene", "Datenbankdesign", "Tableau", "Linux Administration"],
    "Kategorie":   ["Programmierung", "Datenbanken", "Programmierung",
                    "Betriebssysteme", "Business Intelligence", "Programmierung",
                    "Programmierung", "Datenbanken", "Business Intelligence", "Betriebssysteme"],
    "Anmeldungen": [142, 98, 113, 54, 88, 29, 67, 45, 72, 38],
    "Umsatz_EUR":  [113258, 58702, 112887, 26946, 87912, 28971, 66933, 26955, 71928, 18962],
    "Bewertung":   [4.7, 4.5, 4.9, 4.2, 4.8, 3.8, 4.6, 4.1, 4.4, 4.0],
})

pruefungen.head()

,Teilnehmer,Kurs,Kursformat,Trainer,Monat,Note,Bestanden
0,Mia Schulz,Python Grundlagen,Präsenz,Sandra Meier,Januar,2.0,True
1,Jonas Weber,SQL für Einsteiger,Online,Tobias Lange,Januar,1.7,True
2,Lena Koch,Python Grundlagen,Präsenz,Sandra Meier,Januar,3.3,True
3,Tim Bauer,Data Analysis,Online,Michael Brandt,Februar,1.0,True
4,Sara Yilmaz,Linux Basics,Präsenz,Yara Özcan,Februar,2.7,True


### Übung 11 – Prüfungsnoten zusammenfassen (zu Block B)

Arbeite mit `pruefungen`. In Deutschland ist 1,0 die beste Note.

- **a)** Was war die beste Note in allen Prüfungen?
- **b)** Wie viele Prüfungen wurden nicht bestanden?
- **c)** Berechne Mittelwert und Median der Noten.

In [65]:
# a)


In [66]:
# b)


In [67]:
# c)


### Übung 12 – Prüfungen nach Gruppen auswerten (zu Block D)

Arbeite mit `pruefungen`.

- **a)** Berechne die durchschnittliche Note pro Kurs.
- **b)** Ermittle für jeden Trainer die Anzahl der Prüfungen und die Anzahl der bestandenen Prüfungen. Nutze benannte Aggregationen.
- **c)** Berechne für jedes Kursformat die Bestandsquote in Prozent. *Hinweis: Der Mittelwert einer Spalte mit `True` und `False` ist der Anteil der `True`-Werte.*

In [68]:
# a)


In [69]:
# b)


In [70]:
# c)


### Übung 13 – Die Monatsauswertung (zu Block D)

Erstelle aus `pruefungen` einen Bericht pro Monat mit der Anzahl der Prüfungen, der Durchschnittsnote und dem Anteil bestandener Prüfungen. Die Monate sollen in der Reihenfolge Januar, Februar, März, April stehen.

*Hinweis: `groupby` sortiert die Gruppen alphabetisch. Mit `.reindex([...])` bringst du die Zeilen in eine vorgegebene Reihenfolge.*

In [71]:
# Übung 13


### Übung 14 – Die besten und schwächsten Kurse (zu Block E)

Arbeite mit `kursstatistik`.

- **a)** Zeige die 3 Kurse mit den meisten Anmeldungen.
- **b)** Zeige die 2 Kurse mit der schlechtesten Bewertung. Nutze `sort_values()` und `head()`.
- **c)** Welche Kategorie hat insgesamt die meisten Anmeldungen?

In [72]:
# a)


In [73]:
# b)


In [74]:
# c)


---
## 21 – Zusatz: Mehr aus `groupby()`, `agg()` und Top-N

Die Beispiele arbeiten mit `pruefungen` und `kursstatistik` aus Abschnitt 20. Führe die Zelle dort zuerst aus.

**Einfache Aggregationen auf einer Spalte**

| Funktion | Beschreibung |
|---|---|
| `.mean()`, `.median()` | Mittelwert, Median |
| `.sum()`, `.count()` | Summe, Anzahl nicht-leerer Werte |
| `.min()`, `.max()` | kleinster, größter Wert |
| `.std()` | Standardabweichung |
| `.nunique()` | Anzahl verschiedener Werte |
| `.unique()` | die verschiedenen Werte selbst |
| `.value_counts()` | Häufigkeit jedes Wertes |

In [75]:
print("Anzahl Kurse:", pruefungen["Kurs"].nunique())
print("Kurse:", pruefungen["Kurs"].unique().tolist())
print()
print(pruefungen["Kursformat"].value_counts())

Anzahl Kurse: 4
Kurse: ['Python Grundlagen', 'SQL für Einsteiger', 'Data Analysis', 'Linux Basics']

Kursformat
Online     8
Präsenz    7
Name: count, dtype: int64


**Spalten nach `agg()` umbenennen**

`agg()` mit einer Liste benennt die Spalten nach den Funktionen. Mit `.columns = [...]` gibst du ihnen eigene Namen, in derselben Reihenfolge.

In [76]:
note_stats = pruefungen.groupby("Kurs")["Note"].agg(["mean", "min", "max", "count"])
note_stats.columns = ["Durchschnitt", "Beste", "Schlechteste", "Anzahl"]
print(note_stats.round(2))

                    Durchschnitt  Beste  Schlechteste  Anzahl
Kurs                                                         
Data Analysis               1.32    1.0           1.7       4
Linux Basics                3.18    2.0           4.3       4
Python Grundlagen           3.00    2.0           4.0       4
SQL für Einsteiger          2.00    1.7           2.3       3


**Nach zwei Spalten gruppieren**

Mit einer Liste in `groupby()` bildest du Gruppen aus zwei Merkmalen. Das Ergebnis hat einen **MultiIndex**: zwei Ebenen links.

In [77]:
note_kurs_format = pruefungen.groupby(["Kurs", "Kursformat"])["Note"].mean().round(2)
print(note_kurs_format)

Kurs                Kursformat
Data Analysis       Online        1.33
                    Präsenz       1.30
Linux Basics        Online        4.00
                    Präsenz       2.35
Python Grundlagen   Online        4.00
                    Präsenz       2.67
SQL für Einsteiger  Online        2.00
                    Präsenz       2.00
Name: Note, dtype: float64


Lesbarer wird es mit `.unstack()`: Die zweite Ebene wandert in die Spalten.

In [78]:
print(note_kurs_format.unstack())

Kursformat          Online  Präsenz
Kurs                               
Data Analysis         1.33     1.30
Linux Basics          4.00     2.35
Python Grundlagen     4.00     2.67
SQL für Einsteiger    2.00     2.00


**Eigene Berechnung mit `lambda` in `agg()`**

Statt `"mean"` oder `"sum"` kannst du eine eigene kleine Funktion übergeben. `x` steht für die Werte einer Gruppe.

In [79]:
trainer_stats = pruefungen.groupby("Trainer").agg(
    pruefungen=("Note", "count"),
    durchschnittsnote=("Note", "mean"),
    bestandsquote_pct=("Bestanden", lambda x: round(x.mean() * 100, 1)),
)
print(trainer_stats.round(2))

                pruefungen  durchschnittsnote  bestandsquote_pct
Trainer                                                         
Michael Brandt           4               1.32              100.0
Sandra Meier             4               3.00               75.0
Tobias Lange             3               2.00              100.0
Yara Özcan               4               3.18               50.0


**Top-N in zwei Schritten: die Entscheidungshilfe**

Welche 3 Kurse soll die Academy ausbauen? Sie sollen gut bewertet **und** gefragt sein. Erst die 5 besten Bewertungen, daraus die 3 mit den meisten Anmeldungen.

In [80]:
kandidaten = kursstatistik.nlargest(5, "Bewertung")
ausbau = kandidaten.nlargest(3, "Anmeldungen")
print(ausbau[["Kursname", "Anmeldungen", "Bewertung"]])

             Kursname  Anmeldungen  Bewertung
0   Python Grundlagen          142        4.7
2       Data Analysis          113        4.9
1  SQL für Einsteiger           98        4.5


Power BI hat mit 4,8 die zweitbeste Bewertung, fällt im zweiten Schritt aber heraus: 88 Anmeldungen reichen nicht für die Top 3.

**Top-N pro Gruppe**

Die Top-Einträge **innerhalb jeder Kategorie**: erst absteigend sortieren, dann nach Kategorie gruppieren und mit `head(N)` die ersten N Zeilen jeder Gruppe nehmen.

In [81]:
# Umsatzstärkster Kurs pro Kategorie
top1_pro_kategorie = (
    kursstatistik
    .sort_values("Umsatz_EUR", ascending=False)
    .groupby("Kategorie")
    .head(1)
)
print(top1_pro_kategorie[["Kategorie", "Kursname", "Umsatz_EUR"]])

               Kategorie            Kursname  Umsatz_EUR
0         Programmierung   Python Grundlagen      113258
4  Business Intelligence            Power BI       87912
1            Datenbanken  SQL für Einsteiger       58702
3        Betriebssysteme        Linux Basics       26946


In [82]:
# Top 2 nach Anmeldungen pro Kategorie, zum Lesen nach Kategorie sortiert
top2_pro_kategorie = (
    kursstatistik
    .sort_values("Anmeldungen", ascending=False)
    .groupby("Kategorie")
    .head(2)
    .sort_values(["Kategorie", "Anmeldungen"], ascending=[True, False])
)
print(top2_pro_kategorie[["Kategorie", "Kursname", "Anmeldungen"]])

               Kategorie              Kursname  Anmeldungen
3        Betriebssysteme          Linux Basics           54
9        Betriebssysteme  Linux Administration           38
4  Business Intelligence              Power BI           88
8  Business Intelligence               Tableau           72
1            Datenbanken    SQL für Einsteiger           98
7            Datenbanken       Datenbankdesign           45
0         Programmierung     Python Grundlagen          142
2         Programmierung         Data Analysis          113


> **Achtung:** In älteren Anleitungen steht dafür `groupby("Kategorie").apply(lambda x: x.nlargest(1, "Umsatz_EUR"))`. Ab pandas 3 fehlt bei diesem Weg die Spalte `Kategorie` im Ergebnis. Nimm `sort_values()` + `groupby()` + `head()`.

### Übung 15 – Tech oder Non-Tech? (zu Block E)

Arbeite mit `kursstatistik`. Programmierung und Datenbanken zählen als **Tech**, alle anderen Kategorien als **Non-Tech**.

- **a)** Lege eine neue Spalte `Gruppe` mit den Werten `"Tech"` und `"Non-Tech"` an. *Hinweis: `np.where(bedingung, wert_wenn_true, wert_wenn_false)` und `.isin([...])`. Vorher `import numpy as np`.*
- **b)** Wie viele Kurse gehören zu jeder Gruppe?
- **c)** Welcher Kurs hat in jeder Gruppe die meisten Anmeldungen?
- **d)** Erstelle einen Bericht pro Gruppe: Anzahl Kurse, Summe der Anmeldungen, Summe des Umsatzes, mittlere Bewertung.

In [83]:
# a)

In [84]:
# b)

In [85]:
# c)

In [86]:
# d)

### Übung 16 – Sind Online-Kurse schlechter? (zu Block D)

<img src="png%20Dateien/herr_bauchgefuehl.png" alt="Herr Bauchgefühl" width="150">

Herr Bauchgefühl: *„Online lernt doch keiner richtig. Die Noten sind da garantiert schlechter.“*

Arbeite mit `pruefungen`.

- **a)** Wie viele Prüfungen gibt es pro Kurs und Kursformat? Zeige das Ergebnis als Tabelle mit `.unstack()`.
- **b)** Vergleiche Mittelwert, Median und Anzahl der Noten für Online und Präsenz.
- **c)** Berechne die Durchschnittsnote pro Kurs und Kursformat als Tabelle.
- **d)** Hat Herr Bauchgefühl recht? Begründe mit deinen Ergebnissen aus a) bis c).

In [87]:
# a)

In [88]:
# b)

In [89]:
# c)

In [90]:
# d)

---
## 22 – Zusammenfassung

| Aufgabe | Befehl |
|---|---|
| Modus | `mode()`, einzelner Wert mit `mode()[0]` |
| Modalklasse | `pd.cut(...)` und danach `value_counts()` oder `mode()` |
| Median | `median()` |
| Median bei ordinalen Merkmalen | `sort_values()`, `reset_index(drop=True)`, `iloc` |
| Mittelwert | `mean()`, von Hand `sum()` geteilt durch `count()` |
| Runden | `round(wert, 1)` oder `.round(1)` |
| Überblick | `describe()` |
| Gruppen | `groupby("spalte")["wert"].mean()` |
| Mehrere Kennzahlen | `.agg(["count", "mean", "median"])` |
| Bericht mit eigenen Namen | `.agg(name=("spalte", "berechnung"))` |
| Spalten umbenennen | `.columns = ["Name1", "Name2", ...]` |
| Nach zwei Spalten gruppieren | `groupby(["spalte1", "spalte2"])`, als Tabelle mit `.unstack()` |
| Eigene Berechnung in `agg()` | `.agg(name=("spalte", lambda x: ...))` |
| Verschiedene Werte | `.nunique()`, `.unique()` |
| Top-N | `sort_values(...).head(N)`, `nlargest(N, "spalte")`, `nsmallest(N, "spalte")` |
| Top-N in zwei Schritten | `df.nlargest(5, "a").nlargest(3, "b")` |
| Top-N pro Gruppe | `sort_values(..., ascending=False).groupby("g").head(N)` |
| Neue Spalte mit zwei Werten | `np.where(df["s"].isin([...]), "ja", "nein")` |
| Gleichstand | `keep="all"` |

**Welches Lagemaß wann?**

| Skalenniveau | Modus | Median | Mittelwert |
|---|---|---|---|
| nominal | ja | nein | nein |
| ordinal | ja | ja | nein |
| metrisch | ja | ja | ja |

Bei schiefen Verteilungen oder Ausreißern nennst du den Median, am besten zusammen mit dem Mittelwert.

---
*Material erstellt von Julia Brutskaya*